In [12]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

test_df = pd.read_csv(data_path)
test_df.head()

C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv


,id,text
0,00353fb1c541,"The more efficiently we use a resource, the mo..."
1,005f5a9a6139,"The Stern-Gerlach experiment, performed in 192..."
2,0062a53d5487,"Hundreds of millions of years ago, fluctuating..."


In [13]:
import pickle
path = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\deberta_v3_finetuned\le.pkl"
with open(path, "rb") as f:
    le = pickle.load(f)

In [14]:
from transformers import ( AutoModelForSequenceClassification, AutoTokenizer)

LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\deberta_v3_finetuned"
NUM_LABELS = 18
MODEL_NAME = "microsoft/deberta-v3-small"


tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_DIR
    #    MODEL_NAME, tokends=HF_TOKEN
    )

model = AutoModelForSequenceClassification.from_pretrained(
         LOCAL_DIR
        # MODEL_NAME,
        # num_labels=NUM_LABELS,
        # torch_dtype=torch.float32
    )

# model.save_pretrained(LOCAL_DIR)
# tokenizer.save_pretrained(LOCAL_DIR)
model


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 6043.75it/s]


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNor

In [15]:
text = test_df["text"].tolist()
text

['The more efficiently we use a resource, the more of it we ultimately consume: cheaper energy from efficiency gains invites expanded use, eroding the very savings sought. Thus, engineering restraint through efficiency backfires, fueling greater demand, proving that doing more with less often means using far more overall.',
 'The Stern-Gerlach experiment, performed in 1922 by Otto Stern and Walther Gerlach, sent a narrow beam of neutral silver atoms through a strongly nonuniform magnetic field and onto a detector. Classical physics predicted a continuous smear, because randomly oriented magnetic moments should be deflected by all possible amounts. Instead, the beam split into two sharply separated traces. This showed that the atoms’ magnetic moments, and thus their angular momentum, could take only discrete orientations relative to the field. Later understood as evidence for the electron’s intrinsic spin, the result became a landmark of quantum mechanics: microscopic systems do not mer

In [16]:
import torch

encoded = tokenizer(
            text, 
            padding=True,
            truncation=True,
            max_length = 256,
            return_tensors = "pt"
        )

model.eval()
with torch.no_grad():
    outputs = model(
        input_ids = encoded["input_ids"],
        attention_mask = encoded["attention_mask"]
    )
    logits = outputs.logits
    
prob = torch.softmax(logits, dim=1)
pred = torch.argmax(prob, dim=1)

print(pred)
print(le.inverse_transform(pred))

tensor([ 2, 12,  5])
['claude-opus-4-8' 'gpt-5.6-luna' 'gemini-3.1-pro-preview']


In [ ]:
print(le.inverse_transform(pred)) 
predictions_f = le.inverse_transform(pred)
# Submission file 
submission = pd.DataFrame({
    "id": test_df["id"],
    "model": predictions_f
})

submission.to_parquet("submission.parquet", index=False)
print(f"Saved submission.parquet with {len(submission)} rows.")
submission.head(10)

['claude-opus-4-8' 'gpt-5.6-luna' 'gemini-3.1-pro-preview']


In [ ]:
# from psycopg.generators import fetch
# import re
# import numpy as np
# import pandas as pd
# import nltk
# nltk.download("punkt")
# from nltk.tokenize import sent_tokenize, word_tokenize

# features = pd.DataFrame(index=df.index)
# features["model"] = df["model"]
# text_col = "text"

In [ ]:
# # Character, word, and sentence counts
# features["char_count"] = df[text_col].apply(len)
# # Words & Sentences
# def get_words_and_sents(text):
#         sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
#         words = word_tokenize(text) if len(text.strip()) > 0 else [""]
#         return len(words), len(sents), sents, words

# tokenized_info = df[text_col].apply(get_words_and_sents)
# features["word_count"] = tokenized_info.apply(lambda x: x[0])
# features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# # Sentence Length Mean & Variance
# def sent_stats(info):
#     _, num_sents, sents, _ = info
#     if num_sents == 0:
#         return 0.0, 0.0
#     lengths = [len(s.split()) for s in sents]
#     return float(np.mean(lengths)), float(np.var(lengths))

# sent_metrics = tokenized_info.apply(sent_stats)
# features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
# features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# # Avg Word Length
# features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# # ----------------- 2. Markdown Stylometry -----------------
# features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# # ----------------- 3. Punctuation Fingerprint -----------------
# features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
# features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
# features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
# features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
# features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
# features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
# features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
# features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
# features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
# features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# # Punctuation Density (normalized by word count)
# total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
#                 features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
# features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# # ----------------- 4. Capitalization & Casing -----------------
# features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
# features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
# features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
# features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# # ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
# def type_token_ratio(info):
#     num_words, _, _, words = info
#     if num_words == 0:
#         return 0.0
#     unique_words = set(w.lower() for w in words if w.isalnum())
#     return len(unique_words) / (num_words + 1e-5)

# features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# # Digits / Numbers presence
# features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
# features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
# features.head()

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder, StandardScaler

# #  Train-Test Split (stratified)
# df_train, df_test = train_test_split(features, test_size=0.2, random_state=42, stratify=features["model"])
# y_train = df_train["model"].values
# y_test = df_test["model"].values
# X_train_pre = df_train.iloc[:, 1:]
# X_test_pre = df_test.iloc[:, 1:]

# X_train_pre.shape, X_test_pre.shape
# # X_train.head()
# encoder = StandardScaler()
# X_train = encoder.fit_transform(X_train_pre)
# X_test = encoder.transform(X_test_pre)
# X_train.shape, X_test.shape
# X_train

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
# clf = LogisticRegression(max_iter=1000, C=2.0, verbose=1)

# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)

# print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
# print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# # print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")



In [ ]:
# test = pd.read_csv(r"/kaggle/input/competitions/name-that-llm/test.csv")
# test.head()

# features = pd.DataFrame(index=df.index)
# # Character, word, and sentence counts
# features["char_count"] = test[text_col].apply(len)
# # Words & Sentences
# def get_words_and_sents(text):
#         sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
#         words = word_tokenize(text) if len(text.strip()) > 0 else [""]
#         return len(words), len(sents), sents, words

# tokenized_info = test[text_col].apply(get_words_and_sents)
# features["word_count"] = tokenized_info.apply(lambda x: x[0])
# features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# # Sentence Length Mean & Variance
# def sent_stats(info):
#     _, num_sents, sents, _ = info
#     if num_sents == 0:
#         return 0.0, 0.0
#     lengths = [len(s.split()) for s in sents]
#     return float(np.mean(lengths)), float(np.var(lengths))

# sent_metrics = tokenized_info.apply(sent_stats)
# features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
# features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# # Avg Word Length
# features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# # ----------------- 2. Markdown Stylometry -----------------
# features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# # ----------------- 3. Punctuation Fingerprint -----------------
# features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
# features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
# features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
# features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
# features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
# features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
# features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
# features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
# features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
# features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# # Punctuation Density (normalized by word count)
# total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
#                 features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
# features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# # ----------------- 4. Capitalization & Casing -----------------
# features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
# features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
# features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
# features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# # ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
# def type_token_ratio(info):
#     num_words, _, _, words = info
#     if num_words == 0:
#         return 0.0
#     unique_words = set(w.lower() for w in words if w.isalnum())
#     return len(unique_words) / (num_words + 1e-5)

# features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# # Digits / Numbers presence
# features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
# features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
# features.shape

In [ ]:
# X_test_t = features.iloc[:, :].copy()
# X_test = encoder.transform(X_test_t)
# X_test_scaled_df = pd.DataFrame(X_test, columns=X_test_t.columns, index=X_test_t.index)
# X_test_scaled_df

In [ ]:

# predictions = clf.predict(X_test)
# print(predictions)
# print(le.inverse_transform(predictions)) 
# predictions_f = le.inverse_transform(predictions)
# # Submission file 
# submission = pd.DataFrame({
#     "id": test["id"],
#     "model": predictions_f
# })

# submission.to_parquet("submission.parquet", index=False)
# print(f"Saved submission.parquet with {len(submission)} rows.")
# submission.head(10)